In [1]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

/opt/anaconda3/envs/atlas/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
593,"Now I love Bela Lugosi,don't get me wrong,he i...",negative
103,I am a fan of the paranormal and I love Ghost ...,negative
959,I may be a sentimentalist. But i found this mo...,positive
349,"I grew up watching Inspector Gadget. It was, a...",negative
574,I have to say this is the worst movie that I h...,negative


In [3]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [4]:
df = normalize_text(df)
df.head()

,review,sentiment
593,love bela lugosi don t get wrong he one intere...,negative
103,fan paranormal love ghost hunter show first ca...,negative
959,may sentimentalist found movie truly moving fi...,positive
349,grew watching inspector gadget wa still is one...,negative
574,say worst movie ever watched life cannot belie...,negative


In [5]:
df['sentiment'].value_counts()

sentiment
negative    264
positive    236
Name: count, dtype: int64

In [6]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [7]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
593,love bela lugosi don t get wrong he one intere...,0
103,fan paranormal love ghost hunter show first ca...,0
959,may sentimentalist found movie truly moving fi...,1
349,grew watching inspector gadget wa still is one...,0
574,say worst movie ever watched life cannot belie...,0


In [8]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [ ]:
vectorizer = CountVectorizer(max_features=50)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [15]:

# mlflow.set_experiment("Logistic Regression Baseline")


import dagshub
dagshub.init(repo_owner='iamanshchourasiya', repo_name='Capstone-Project', mlflow=True)

mlflow.set_experiment("Logistic Regression Baseline")

2026-07-18 15:36:23,849 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/iamanshchourasiya/Capstone-Project "HTTP/1.1 200 OK"


Initialized MLflow to track repo "iamanshchourasiya/Capstone-Project"

2026-07-18 15:36:23,852 - INFO - Initialized MLflow to track repo "iamanshchourasiya/Capstone-Project"


Repository iamanshchourasiya/Capstone-Project initialized!

2026-07-18 15:36:23,853 - INFO - Repository iamanshchourasiya/Capstone-Project initialized!


<Experiment: artifact_location='mlflow-artifacts:/f06efd2fb9b742f2874ebb88d23c5da5', creation_time=1784368234822, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1784368234822, lifecycle_stage='active', name='Logistic Regression Baseline', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [16]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 50)
        mlflow.log_param("test_size", 0.3)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-07-18 15:36:28,482 - INFO - Starting MLflow run...
2026-07-18 15:36:28,948 - INFO - Logging preprocessing parameters...
2026-07-18 15:36:29,883 - INFO - Initializing Logistic Regression model...
2026-07-18 15:36:29,883 - INFO - Fitting the model...
2026-07-18 15:36:29,889 - INFO - Model training complete.
2026-07-18 15:36:29,889 - INFO - Logging model parameters...
2026-07-18 15:36:30,214 - INFO - Making predictions...
2026-07-18 15:36:30,215 - INFO - Calculating evaluation metrics...
2026-07-18 15:36:30,220 - INFO - Logging evaluation metrics...
2026-07-18 15:36:31,444 - INFO - Saving and logging the model...
2026/07/18 15:36:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-07-18 15:36:39,558 - INFO - Model training and logging completed in 10.61 seconds.
2026-07-18 15:36:39,559 - INFO - Accuracy: 0.6866666666666666
2026-07-18 15:36:39,559 - INFO - Precision: 0.6865671641791045
2026-07-18 15:36:39,560 - INFO - Recall: 0.63888888888888

🏃 View run rebellious-cow-557 at: https://dagshub.com/iamanshchourasiya/Capstone-Project.mlflow/#/experiments/0/runs/be97571515fa48668387d3cd5b7f2d0a
🧪 View experiment at: https://dagshub.com/iamanshchourasiya/Capstone-Project.mlflow/#/experiments/0
